# VILA Model Evaluation: Reproducing Table 1 Results

## Overview
This notebook reproduces **Table 1** from the VILA (Vision and Language Aesthetic) research paper by evaluating two VILA models on the AVA (Aesthetic Visual Assessment) dataset. The notebook demonstrates how VILA-Pretrain (VILA-P) and VILA-Rank (VILA-R) perform on aesthetic quality prediction tasks.

## Objective
Reproduce and validate the reported performance metrics (SRCC and PLCC) for:
- **VILA-P (Zero-Shot Learning)**: Pretrained model using text prompts for quality assessment
- **VILA-R (Rank-Tuned)**: Fine-tuned model specifically optimized for ranking aesthetic quality

## Dataset
- **Name**: AVA (Aesthetic Visual Assessment)
- **Source**: Kaggle (`nicolacarrassi/ava-aesthetic-visual-assessment`)
- **Size**: ~255K images with aesthetic ratings
- **Features**:
  - Image quality ratings (1-10) from multiple annotators
  - Mean Opinion Score (MOS) computed from vote distributions
  - Standard deviation of ratings
  - Tags and challenge IDs

## Methodology

### Metrics
- **SRCC** (Spearman Rank Correlation Coefficient): Measures monotonic relationship
- **PLCC** (Pearson Linear Correlation Coefficient): Measures linear correlation

## Technical Setup

### Environment
- **Platform**: Google Colab with A100 GPU (or lower)
- **Custom Environment**: Conda environment (`vila_env`) with Python 3.9

### Model Checkpoints
Downloaded from Google Research cloud storage:
- VILA-Pretrain checkpoint
- VILA-Rank-tuned checkpoint
- SentencePiece tokenizer model

### Key Dependencies
- `lingvo==0.12.6`, `paxml==1.0.0`
- `jax==0.4.7`, `jaxlib==0.4.7`
- Image processing: PIL, torchvision
- Data handling: pandas, numpy

## ⚡ Performance Optimizations
- **Parallel Processing**: ThreadPoolExecutor with configurable workers (8-12 threads)
- **Crash Recovery**: Results saved incrementally to CSV to resume after interruptions
- **Thread-Safe I/O**: CSV writes protected with locks for concurrent access

##  Expected Results

| Model           | SRCC (Paper) | PLCC (Paper) |
|-----------------|--------------|--------------|
| VILA-P (ZSL)    | 0.657        | 0.663        |
| VILA-R          | 0.774        | 0.774        |

The notebook computes actual SRCC and PLCC values and compares them with paper results (Δ metrics).


---
**Author**: Reproducing results from Google Research VILA paper Łukasz Popek
**Last Updated**: 2026  
**Runtime**: Google Colab (High-RAM, A100 GPU recommended)


In [ ]:
!pip install -q kaggle

from google.colab import files
print("Upload kaggle.json (download from: kaggle.com → Settings → API → Create New Token)")
uploaded = files.upload()  # upload kaggle.json

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print("Kaggle API configured!")

Upload kaggle.json (download from: kaggle.com → Settings → API → Create New Token)


Saving kaggle.json to kaggle.json
Kaggle API configured!


In [ ]:
!kaggle datasets download \
    -d nicolacarrassi/ava-aesthetic-visual-assessment \
    -p /content/EEML_VILA \
    --unzip

Dataset URL: https://www.kaggle.com/datasets/nicolacarrassi/ava-aesthetic-visual-assessment
License(s): unknown
100% 31.0G/31.0G [13:47<00:00, 40.2MB/s]



In [ ]:
PROJECT_PATH = '/content/EEML_VILA'

In [ ]:
import os
for root, dirs, files_ in os.walk(PROJECT_PATH):
    dirs[:] = [d for d in dirs if d != 'images']
    for f in files_:
        full = os.path.join(root, f)
        size = os.path.getsize(full) / 1e6
        print(f"{full}  ({size:.1f} MB)")

/content/EEML_VILA/ground_truth_dataset.csv  (47.9 MB)
/content/EEML_VILA/AVA_Files/README.txt  (0.0 MB)
/content/EEML_VILA/AVA_Files/tags.txt  (0.0 MB)
/content/EEML_VILA/AVA_Files/AVA.txt  (12.5 MB)
/content/EEML_VILA/AVA_Files/challenges.txt  (0.0 MB)


In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

KAGGLE_DIR  = '/content/EEML_VILA'
IMAGES_DIR  = os.path.join(KAGGLE_DIR, 'images')
AVA_TXT     = os.path.join(KAGGLE_DIR, 'AVA_Files', 'AVA.txt')
TAGS_TXT    = os.path.join(KAGGLE_DIR, 'AVA_Files', 'tags.txt')
CHALLENGES  = os.path.join(KAGGLE_DIR, 'AVA_Files', 'challenges.txt')

all_images = os.listdir(IMAGES_DIR)
print(f"Images on disk: {len(all_images):,}")
print(f"Examples: {all_images[:5]}")

Images on disk: 255,508
Examples: ['132336.jpg', '176595.jpg', '702163.jpg', '87014.jpg', '921816.jpg']


In [ ]:
cols = (
    ['index', 'image_id'] +
    [f'rating_{i}' for i in range(1, 11)] +
    ['tag1', 'tag2', 'challenge_id']
)

ava_df = pd.read_csv(AVA_TXT, sep=' ', header=None, names=cols)

# Compute Mean Opinion Score (MOS) and standard deviation from raw vote counts
ratings     = np.arange(1, 11)
vote_matrix = ava_df[[f'rating_{i}' for i in range(1, 11)]].values
total_votes = vote_matrix.sum(axis=1)
ava_df['mos'] = (vote_matrix * ratings).sum(axis=1) / total_votes
ava_df['std'] = np.sqrt(
    ((vote_matrix * (ratings - ava_df['mos'].values[:, None])**2)
     .sum(axis=1)) / total_votes
)

# Load tag and challenge lookup dictionaries
tags, challenges = {0: 'None'}, {}
with open(TAGS_TXT) as f:
    for line in f:
        p = line.strip().split(' ', 1)
        if len(p) == 2: tags[int(p[0])] = p[1]
with open(CHALLENGES) as f:
    for line in f:
        p = line.strip().split(' ', 1)
        if len(p) == 2: challenges[int(p[0])] = p[1]

ava_df['tag1_name']      = ava_df['tag1'].map(tags)
ava_df['tag2_name']      = ava_df['tag2'].map(tags)
ava_df['challenge_name'] = ava_df['challenge_id'].map(challenges)
ava_df['img_path']       = ava_df['image_id'].apply(
    lambda x: os.path.join(IMAGES_DIR, f'{x}.jpg')
)

# Keep only images that are actually available on disk
existing = set(int(f.replace('.jpg','')) for f in all_images if f.endswith('.jpg'))
ava_df   = ava_df[ava_df['image_id'].isin(existing)]

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    ava_df,
    test_size=19928,
    random_state=42
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df):,}")
print(f"Test:  {len(test_df):,}")

Train: 235,580
Test:  19,928


In [ ]:
import torchvision.transforms as T

class AVADataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df
        self.transform = transform or T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize([0.5, 0.5, 0.5],
                        [0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['img_path']).convert('RGB')
        return {
            'image':          self.transform(img),
            'image_pil':      img,
            'image_id':       int(row['image_id']),
            'mos':            float(row['mos']),
            'std':            float(row['std']),
            'tag1_name':      row['tag1_name'],
            'challenge_name': row['challenge_name'],
        }

train_dataset = AVADataset(train_df)
test_dataset  = AVADataset(test_df)

sample = test_dataset[0]
print(f"image_id:  {sample['image_id']}")
print(f"MOS:       {sample['mos']:.3f}")
print(f"tag:       {sample['tag1_name']}")
print(f"challenge: {sample['challenge_name']}")
print(f"tensor:    {sample['image'].shape}")
print(f"PIL:       {sample['image_pil']}")

image_id:  342907
MOS:       6.118
tag:       Nature
challenge: Green_II
tensor:    torch.Size([3, 224, 224])
PIL:       <PIL.Image.Image image mode=RGB size=640x384 at 0x7AC8DC98CE60>


In [ ]:
train_df.to_csv(os.path.join(KAGGLE_DIR, 'train_split.csv'), index=False)
test_df.to_csv(os.path.join(KAGGLE_DIR, 'test_split.csv'),  index=False)
print("[INFO]: Saved to train_split.csv i test_split.csv")

[INFO]: Saved to train_split.csv i test_split.csv


In [ ]:
!git clone https://github.com/google-research/google-research.git

Cloning into 'google-research'...
remote: Enumerating objects: 121608, done.
remote: Counting objects: 100% (385/385), done.
remote: Compressing objects: 100% (295/295), done.
remote: Total 121608 (delta 222), reused 92 (delta 89), pack-reused 121223 (from 6)
Receiving objects: 100% (121608/121608), 1.13 GiB | 36.25 MiB/s, done.
Resolving deltas: 100% (73377/73377), done.
Updating files: 100% (21821/21821), done.


In [ ]:
print("Checking available checkpoints...")
!gsutil ls gs://gresearch/vila/checkpoints/

print("\nDownloading all checkpoints...")
!mkdir -p /content/checkpoints/vila
!gsutil -m cp -r gs://gresearch/vila/checkpoints/* /content/checkpoints/vila/

print("\nDownloaded file structure:")
!ls -lh /content/checkpoints/vila/
!echo "\nContents of each subfolder:"
!find /content/checkpoints/vila/ -maxdepth 2 -type d

Checking available checkpoints...
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://gresearch/vila/checkpoints/laion_pretrain/
gs://gresearch/vila/checkpoints/vila_pretrain/
gs://gresearch/vila/checkpoints/vila_rank_tuned/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://gresearch/vila/checkpoints/laion_pretrain/checkpoint_0/metadata/metadata...
Copying gs://gresearch/vila/checkpoints/laion_pretrain/checkpoint_0/state/checkpoint...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object dow

In [ ]:
!mkdir -p /content/checkpoints/vila/spm_model/
!ls -lah /content/checkpoints/vila/spm_model/
!gsutil -m cp -r gs://gresearch/vila/spm_model/* /content/checkpoints/vila/spm_model/

total 8.0K
drwxr-xr-x 2 root root 4.0K Mar 29 12:26 .
drwxr-xr-x 6 root root 4.0K Mar 29 12:26 ..
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://gresearch/vila/spm_model/spm.model...
Copying gs://gresearch/vila/spm_model/spm.vocab...
- [2/2 files][  2.5 MiB/  2.5 MiB] 100% Done                                    
Operation completed over 2 objects/2.5 MiB.                                      


In [ ]:
#setting enviroment for VILA
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!bash ./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local

--2026-03-29 12:26:59--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 162067098 (155M) [application/octet-stream]
Saving to: ‘Miniconda3-latest-Linux-x86_64.sh’

Miniconda3-latest-L 100%[===================>] 154.56M   332MB/s    in 0.5s    

2026-03-29 12:27:00 (332 MB/s) - ‘Miniconda3-latest-Linux-x86_64.sh’ saved [162067098/162067098]

PREFIX=/usr/local
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please

In [ ]:
!conda config --set allow_conda_downgrades true
!yes | conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!yes | conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [ ]:
!conda init bash

no change     /usr/local/condabin/conda
no change     /usr/local/bin/conda
no change     /usr/local/bin/conda-env
no change     /usr/local/bin/activate
no change     /usr/local/bin/deactivate
no change     /usr/local/etc/profile.d/conda.sh
no change     /usr/local/etc/fish/conf.d/conda.fish
no change     /usr/local/shell/condabin/Conda.psm1
no change     /usr/local/shell/condabin/conda-hook.ps1
no change     /usr/local/lib/python3.13/site-packages/xontrib/conda.xsh
no change     /usr/local/etc/profile.d/conda.csh
modified      /root/.bashrc

==> For changes to take effect, close and re-open your current shell. <==



In [ ]:
!conda create -n vila_env python=3.9 -y

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / - \ | / - \ | done
Channels:
 - defaults
Platform: linux-64
Solving environment: | done

## Package Plan ##

  environment location: /usr/local/envs/vila_env

  added / updated specs:
    - python=3.9


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    expat-2.7.5                |       h7354ed3_0          24 KB
    ld_impl_linux-64-2.44      |       h9e0c5a2_3         725 KB
    libexpat-2.7.5             |       h7354ed3_0         122 KB
    libnsl-2.0.0               |       h5eee18b_0          31 KB
    python-3.9.25              |       h0dcde21_1        23.0 MB
    setuptools-80.9.0          |   py39h06a4308_0         1.4 MB
    sqlite-3.51.2              |       h3e8d24a_0         1.2 MB
    tzdata-2026a               |       he532380_0         117 KB
    wheel-0.45.1         

In [ ]:
!/usr/local/bin/conda install -n vila_env -c conda-forge \
    "absl-py>=0.12.0" \
    jax==0.4.7 \
    jaxlib==0.4.7 \
    numpy \
    pillow \
    matplotlib \
    scipy \
    tensorflow -y

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: - \ | done

## Package Plan ##

  environment location: /usr/local/envs/vila_env

  added / updated specs:
    - absl-py[version='>=0.12.0']
    - jax==0.4.7
    - jaxlib==0.4.7
    - matplotlib
    - numpy
    - pillow
    - scipy
    - tensorflow


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _tflow_select-2.3.0        |              mkl           2 KB
    absl-py-2.3.1              |     pyhd8ed1ab_0         107 KB  conda-forge
    alsa-lib-1.2.15.3          |       hb03c661_0         571 KB  conda-forge
    astor-0.8.1                |     pyhd8ed1ab_1          29 KB  conda-forge
    astunparse-1.6.3           |     pyhd8ed1ab_3          18 KB  conda-forge
    brotli-1.2.0               |       hed03a55_1          20 KB  conda-forge
    brotli-

In [ ]:
!/usr/local/envs/vila_env/bin/pip install lingvo==0.12.6 paxml==1.0.0

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at m

In [ ]:
!/usr/local/envs/vila_env/bin/pip install numpy==1.23.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 100.9 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
paxml 1.0.0 requires numpy~=1.24.2, but you have numpy 1.23.5 which is incompatible.
praxis 1.0.0 requires numpy~=1.24.2, but you have numpy 1.23.5 which is incompatible.


In [ ]:
!/usr/local/envs/vila_env/bin/pip install flatbuffers==1.12

In [ ]:
!/usr/local/envs/vila_env/bin/pip install tensorflow==2.9.3

In [ ]:
!/usr/local/envs/vila_env/bin/pip install flatbuffers==23.5.26

  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 1.12
    Uninstalling flatbuffers-1.12:
      Successfully uninstalled flatbuffers-1.12
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.9.3 requires flatbuffers<2,>=1.12, but you have flatbuffers 23.5.26 which is incompatible.


In [ ]:
!/usr/local/envs/vila_env/bin/python --version
!/usr/local/envs/vila_env/bin/pip show tensorflow
!/usr/local/envs/vila_env/bin/pip show flatbuffers

Python 3.9.25
Name: tensorflow
Version: 2.9.3
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /usr/local/envs/vila_env/lib/python3.9/site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google-pasta, grpcio, h5py, keras, keras-preprocessing, libclang, numpy, opt-einsum, packaging, protobuf, setuptools, six, tensorboard, tensorflow-estimator, tensorflow-io-gcs-filesystem, termcolor, typing-extensions, wrapt
Required-by: lingvo, paxml, praxis, tensorflow-text
Name: flatbuffers
Version: 23.5.26
Summary: The FlatBuffers serialization format for Python
Home-page: https://google.github.io/flatbuffers/
Author: Derek Bailey
Author-email: derekbailey@google.com
License: Apache 2.0
Location: /usr/local/envs/vila_env/lib/python3.9/site-packages
Requires: 
Required-by: tensorflow


In [ ]:
!/usr/local/envs/vila_env/bin/pip install scipy==1.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.5/38.5 MB 109.1 MB/s  0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.13.1
    Uninstalling scipy-1.13.1:
      Successfully uninstalled scipy-1.13.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
paxml 1.0.0 requires numpy~=1.24.2, but you have numpy 1.23.5 which is incompatible.
praxis 1.0.0 requires numpy~=1.24.2, but you have numpy 1.23.5 which is incompatible.


In [ ]:
!find /content/google-research/vila -name "*.py" -type f -exec sed -i 's/@dataclasses\.dataclass(slots=True)/@dataclasses.dataclass/g' {} +

In [ ]:
%%bash
/usr/local/envs/vila_env/bin/python << 'EOF'
import sys
sys.path.insert(0, '/content/google-research')
from vila import run_vila_predict
print("✅ VILA is woroking!!")
EOF

✅ VILA DZIAŁA!


/usr/local/envs/vila_env/lib/python3.9/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [ ]:

VILA_PREDICT = '/content/google-research/vila/run_vila_predict.py'
!cp {VILA_PREDICT} {VILA_PREDICT}.backup
print("Backup saved.")
with open(VILA_PREDICT, 'r') as f:
    content = f.read()
old = (
    "      zsl_scores = zsl_scores[:, 0]\n"
    "      print('===== VILA ZSL predicted quality score [0, 1]: ', zsl_scores)"
)
new = (
    "      # --- Single prompt: only first pair ('good image', 'bad image') ---\n"
    "      zsl_single = jnp.matmul(image_embed, text_embed[:2].T)\n"
    "      zsl_single = zsl_single.reshape([-1, 1, 2])\n"
    "      zsl_single = jax.nn.softmax(zsl_single).mean(1)[:, 0]\n"
    "      print('===== VILA ZSL single prompt score [0, 1]: ', zsl_single)\n\n"
    "      # --- Ensemble: all 6 prompt pairs ---\n"
    "      zsl_ens = jnp.matmul(image_embed, text_embed.T)\n"
    "      zsl_ens = zsl_ens.reshape([-1, len(_ZSL_QUALITY_PROMPTS), 2])\n"
    "      zsl_ens = jax.nn.softmax(zsl_ens).mean(1)[:, 0]\n"
    "      print('===== VILA ZSL ensemble score [0, 1]: ', zsl_ens)"
)

if old in content:
    content = content.replace(old, new)
    with open(VILA_PREDICT, 'w') as f:
        f.write(content)
    print("File patched successfully!")
else:
    print("[ERROR]: Old block not found — check indentation in the file.")

Backup saved.
File patched successfully!


In [ ]:
import subprocess, os, re
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import spearmanr, pearsonr
GOOGLE_RESEARCH = '/content/google-research'
CKPT_VILAR = '/content/checkpoints/vila/vila_rank_tuned'
CKPT_VILAP = '/content/checkpoints/vila/vila_pretrain'
SPM_PATH   = '/content/checkpoints/vila/spm_model/spm.model'
VILA_PYTHON = '/usr/local/envs/vila_env/bin/python'

def run_vila_predict(image_path, ckpt_dir, spm_path):
    result_QC = subprocess.run(
        [
            VILA_PYTHON, '-m', 'vila.run_vila_predict',
            f'--ckpt_dir={ckpt_dir}',
            f'--image_path={image_path}',
            f'--spm_model_path={spm_path}',
        ],
        capture_output=True,
        text=True,
        cwd=GOOGLE_RESEARCH,
    )
    quality_score, zsl_single_score, zsl_ensemble_score = None, None, None
    if result_QC.returncode != 0:
        print(f"[ERROR]: {image_path}:\n{result_QC.stderr}")

    for line in result_QC.stdout.splitlines():
        if 'VILA predicted quality score' in line:
            nums = re.findall(r'[\d.]+', line.split(':')[-1])
            if nums:
                quality_score = float(nums[0])
        if 'VILA ZSL single prompt score' in line:
            nums = re.findall(r'[\d.]+', line.split(':')[-1])
            if nums:
                zsl_single_score = float(nums[0])
        if 'VILA ZSL ensemble score' in line:
            nums = re.findall(r'[\d.]+', line.split(':')[-1])
            if nums:
                zsl_ensemble_score = float(nums[0])

    if quality_score is None or zsl_single_score is None or zsl_ensemble_score is None:
        print(f"[ERROR]: Not all scores were calculated")
        return None, None, None
    return quality_score, zsl_single_score, zsl_ensemble_score

In [ ]:
test_img = test_df.iloc[0]['img_path']
print(f"Test image: {test_img}")

q_score, zsl_single_score, zsl_ensemble_score = run_vila_predict(test_img, CKPT_VILAR, SPM_PATH)
print(f"Quality score (VILA-R): {q_score}")
print(f"ZSL single score    (VILA-P):  {zsl_single_score}")
print(f"ZSL ensemble score    (VILA-P):  {zsl_ensemble_score}")

Test image: /content/EEML_VILA/images/342907.jpg
Quality score (VILA-R): 0.55705565
ZSL single score    (VILA-P):  0.50146896
ZSL ensemble score    (VILA-P):  0.50137305


In [ ]:
csv_path = os.path.join(RESULTS_DIR, f'processed_data.csv')
print(csv_path)

/content/drive/MyDrive/EEML_VILA/results/processed_data.csv


In [ ]:
import pandas as pd
import os

already_done = set()
if os.path.exists(csv_path):
    done_df      = pd.read_csv(csv_path)
    done_df      = done_df.dropna(subset=['vila_rank'])
    already_done = set(done_df['img_path'].tolist())
    print(f"Already processed: {len(already_done):,} images — skipping these")
else:
    print("No existing results file — starting fresh")

# Filter out already processed rows
rows = [
    row for _, row in df_eval.iterrows()
    if row['img_path'] not in already_done
]
print(f"Remaining to process: {len(rows):,} images")

Already processed: 8,182 images — skipping these
Remaining to process: 11,746 images


In [ ]:
MAX_WORKERS = 12

vilar_scores        = []
zsl_single_scores   = []
zsl_ensemble_scores = []
gt_mos              = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(process_single_image, row): row
        for row in rows
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc=f'VILA Evaluation (workers={MAX_WORKERS})'
    ):
        result = future.result()
        if result is not None:
            q, zsl_s, zsl_e, mos = result
            vilar_scores.append(q)
            zsl_single_scores.append(zsl_s   if zsl_s is not None else 0.0)
            zsl_ensemble_scores.append(zsl_e if zsl_e is not None else 0.0)
            gt_mos.append(mos)

if already_done:
    done_df = pd.read_csv(csv_path)
    done_df = done_df.dropna(subset=['vila_rank'])

    vilar_scores        = np.concatenate([done_df['vila_rank'].values,    np.array(vilar_scores)])
    zsl_single_scores   = np.concatenate([done_df['zsl_single'].values,   np.array(zsl_single_scores)])
    zsl_ensemble_scores = np.concatenate([done_df['zsl_ensemble'].values, np.array(zsl_ensemble_scores)])
    gt_mos              = np.concatenate([done_df['mos_gt'].values,        np.array(gt_mos)])
else:
    vilar_scores        = np.array(vilar_scores)
    zsl_single_scores   = np.array(zsl_single_scores)
    zsl_ensemble_scores = np.array(zsl_ensemble_scores)
    gt_mos              = np.array(gt_mos)

print(f"\n--- RESULTS ---")
print(f"Successfully processed: {len(gt_mos):,} images")
print(f"  — from previous run:  {len(already_done):,}")
print(f"  — from this run:      {len(gt_mos) - len(already_done):,}")
print(f"Saved to:               {csv_path}")

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Thread-safe CSV writer
csv_lock = threading.Lock()

def process_single_image(row):
    """Process one image — called in parallel."""
    q, zsl_s, zsl_e = run_vila_predict(row['img_path'], CKPT_VILAR, SPM_PATH)

    if q is not None:
        # Thread-safe write to CSV
        with csv_lock:
            with open(csv_path, 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([
                    row['img_path'],
                    zsl_s if zsl_s is not None else '',
                    zsl_e if zsl_e is not None else '',
                    q,
                    float(row['mos']),
                ])
        return q, zsl_s, zsl_e, float(row['mos'])
    else:
        print(f"Skipped: {row['img_path']}")
        return None

In [ ]:
if os.path.exists(csv_path):
    done_df      = pd.read_csv(csv_path)
    done_df      = done_df.dropna(subset=['vila_rank'])
    already_done = set(done_df['img_path'].tolist())
    print(f"Already processed: {len(already_done):,} images")
else:
    already_done = set()
    print("No existing results file — starting fresh")

MAX_WORKERS = 8

vilar_scores        = []
zsl_single_scores   = []
zsl_ensemble_scores = []
gt_mos              = []

rows = [row for _, row in df_eval.iterrows()]

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {}
    for row in rows:
        if row['img_path'] in already_done:
            continue
        futures[executor.submit(process_single_image, row)] = row

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc=f'VILA Evaluation (workers={MAX_WORKERS})'
    ):
        result = future.result()
        if result is not None:
            q, zsl_s, zsl_e, mos = result
            vilar_scores.append(q)
            zsl_single_scores.append(zsl_s   if zsl_s is not None else 0.0)
            zsl_ensemble_scores.append(zsl_e if zsl_e is not None else 0.0)
            gt_mos.append(mos)

            # Update set immediately so subsequent checks are aware
            already_done.add(futures[future]['img_path'])

vilar_scores        = np.array(vilar_scores)
zsl_single_scores   = np.array(zsl_single_scores)
zsl_ensemble_scores = np.array(zsl_ensemble_scores)
gt_mos              = np.array(gt_mos)

print(f"\n--- RESULTS ---")
print(f"Newly processed:  {len(gt_mos):,} images")
print(f"Total done so far:{len(already_done):,} images")
print(f"Saved to:         {csv_path}")

Already processed: 8,182 images


VILA Evaluation (workers=8):   0%|          | 0/11746 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
  File "/usr/local/envs/vila_env/lib/python3.9/site-packages/jax/_src/core.py", line 359, in bind
    return self.bind_with_trace(find_top_trace(args), args, params)
  File "/usr/local/envs/vila_env/lib/python3.9/site-packages/jax/_src/core.py", line 362, in bind_with_trace
    out = trace.process_primitive(self, map(trace.full_raise, args), params)
  File "/usr/local/envs/vila_env/lib/python3.9/site-packages/jax/_src/core.py", line 816, in process_primitive
    return primitive.impl(*tracers, **params)
  File "/usr/local/envs/vila_env/lib/python3.9/site-packages/jax/_src/dispatch.py", line 117, in apply_primitive
    compiled_fun = xla_primitive_callable(prim, *unsafe_map(arg_spec, args),
  File "/usr/local/envs/vila_env/lib/python3.9/site-packages/jax/_src/util.py", line 253, in wrapper
    return cached(config._trace_context(), *args, **kwargs)
  File "/usr/local/envs/vila_env/lib/python3.9/site-packages/jax/_src/util.py", line 246,